# 06. Forma del crecimiento (segunda representación)

**Pregunta de partida.** El clustering sobre el nivel (notebooks 04 y 05) separa a los artistas por cuánto crecen: la recta de crecimiento explica el 95% de cada serie y tapa la forma. Entonces, ¿cómo crecen? ¿Hay artistas con explosiones y otros con crecimiento parejo?

**Cambio de representación.** En lugar del nivel se usa la **tasa de crecimiento mensual** (derivada FIR sobre log1p suavizado, notebook 03). La pendiente pasa a ser un valor de fondo y quedan a la vista las aceleraciones, las explosiones y la desaceleración. Una explosión, que en el nivel de una audiencia acumulada es un escalón, en la tasa es un pico.

Etapas: 1, qué variables entran y distancias; 2, clustering de la forma; 3, cuántas formas hay.

In [ ]:
# PAQUETES

library(data.table)
library(cluster)
library(dtwclust)
library(ggplot2)

# Tipografias de la presentacion (se dibujan con ragg): Berlin Sans FB Demi
# para titulos y DejaVu Sans Mono para el resto (copia en fuentes/)
systemfonts::register_font("LWR Titulo", plain = "C:/Windows/Fonts/BRLNSDB.TTF")
systemfonts::register_font(
  "LWR Mono",
  plain = "fuentes/DejaVuSansMono.ttf",
  bold = "fuentes/DejaVuSansMono-Bold.ttf"
)

# dtwclust cambia el generador aleatorio al cargarse: se fija el
# generador por defecto de R para que las semillas sean reproducibles
RNGkind("Mersenne-Twister", "Inversion", "Rejection")

## 1. Configuración y carga

In [ ]:
# ============================================================
# CONFIGURACION
# ============================================================

archivos_panel_preprocesado <- list.files(
  pattern = "^artistas_ST_[0-9]+de[0-9]+_top[0-9]+_22_26_preprocesado_.+_long[.]csv$"
)
stopifnot("No hay panel preprocesado: correr el notebook 03" = length(archivos_panel_preprocesado) > 0)
ARCHIVO_PANEL_PREPROCESADO <- archivos_panel_preprocesado[which.max(file.mtime(archivos_panel_preprocesado))]

VARIABLES_PANEL <- c(
  spotify_followers = "Seguidores Spotify",
  instagram_followers = "Seguidores Instagram",
  youtube_channel_subscribers = "Suscriptores YouTube",
  spotify_playlist_count = "Playlists Spotify",
  youtube_playlist_count = "Playlists YouTube"
)

# Variables de la segunda representación: las tres audiencias
# acumuladas (decisión fundamentada en la sección 2)
VARIABLES_FORMA <- c("spotify_followers", "instagram_followers", "youtube_channel_subscribers")

COLUMNA_SERIE <- "crecimiento_mensual"

# Sin los 2 meses de cada borde: la derivada y la media móvil usan
# un solo lado (misma decisión que en las features del notebook 03)
MESES_BORDE_SIN_EXTREMOS <- 2L

# DTW: misma configuración que el notebook 04
VENTANA_DTW_MESES <- 6L
PATRON_PASOS_DTW <- dtw::symmetric1
NORMA_DTW <- "L2"

ARCHIVO_DISTANCIAS_FORMA <- file.path("chartmetric_pilotos", "distancias_forma_crecimiento.rds")
ARCHIVO_DISTANCIAS_NIVEL <- file.path("chartmetric_pilotos", "distancias_clustering.rds")
FIGURES_DIR <- "figuras"

ARCHIVO_PANEL_PREPROCESADO

In [ ]:
# ============================================================
# CARGA DEL CRECIMIENTO MENSUAL, sin los meses de borde
# ============================================================

panel <- fread(ARCHIVO_PANEL_PREPROCESADO, select = c("artist_id", "artist_name", "variable", "month", COLUMNA_SERIE))
setnames(panel, COLUMNA_SERIE, "crecimiento")
panel[, month := as.Date(month)]

MESES <- sort(unique(panel$month))
MESES_INTERIOR <- MESES[(MESES_BORDE_SIN_EXTREMOS + 1):(length(MESES) - MESES_BORDE_SIN_EXTREMOS)]
panel <- panel[month %in% MESES_INTERIOR]

artistas <- unique(panel[, .(artist_id, artist_name)])
setorder(artistas, artist_id)

cat("Artistas:", nrow(artistas), "| meses:", length(MESES_INTERIOR),
    "(", format(min(MESES_INTERIOR)), "a", format(max(MESES_INTERIOR)), ")\n")

## 2. ¿Qué variables entran?

**Pregunta.** La derivada es un pasa-altos: amplifica el ruido (clase 08). ¿Todas las variables tienen una tasa de crecimiento con señal?

La tabla mide, por variable, el crecimiento mensual típico, la proporción de meses sin crecimiento y cuánto se parece la curva de cada artista a la curva común de todos (correlación con la mediana mensual del panel). Una variable cuya tasa es casi siempre cero, o que se mueve igual para todos, no distingue formas.

In [ ]:
# ============================================================
# DIAGNOSTICO DE LA TASA DE CRECIMIENTO POR VARIABLE
# - crecimiento_tipico_pct: mediana del crecimiento mensual;
# - pct_meses_sin_crecimiento: meses con tasa exactamente 0;
# - correlacion_con_curva_comun: mediana entre artistas de la
#   correlación de su tasa con la mediana mensual del panel
#   (alta = todos se mueven igual, la variable no distingue).
# ============================================================

curva_comun <- panel[, .(mediana_mensual = median(crecimiento)), by = .(variable, month)]
panel_con_comun <- merge(panel, curva_comun, by = c("variable", "month"))

diagnostico_variables <- panel_con_comun[
  ,
  .(
    tipico = median(crecimiento),
    sin_crecimiento = mean(crecimiento == 0),
    correlacion = suppressWarnings(cor(crecimiento, mediana_mensual))
  ),
  by = .(artist_id, variable)
][
  ,
  .(
    crecimiento_tipico_pct = round(100 * median(tipico), 2),
    pct_meses_sin_crecimiento = round(100 * mean(sin_crecimiento), 1),
    correlacion_con_curva_comun = round(median(correlacion, na.rm = TRUE), 2)
  ),
  by = variable
]

diagnostico_variables[, entra := variable %in% VARIABLES_FORMA]
diagnostico_variables[order(-entra, variable)]

**Decisión.** Entran las tres audiencias acumuladas: seguidores de Spotify, seguidores de Instagram y suscriptores de YouTube. Quedan afuera las dos variables de playlists:

- **Playlists de Spotify:** la tasa de cada artista se parece mucho a la curva común (correlación mediana de 0,91): se mueven en olas compartidas, que reflejan a la plataforma más que al artista.
- **Playlists de YouTube:** crecimiento típico casi nulo (0,09% mensual) y la menor correlación con la curva común (0,31): cuentas chicas que saltan de a una playlist, mayormente ruido (17% de energía rápida, notebook 03).

La pregunta pasa a ser cómo crece la **audiencia**.

**¿Por qué centrar también la tasa?** El promedio de la tasa de crecimiento de un artista es su pendiente: la misma intensidad que ya ordenó a la representación del nivel. Sin centrar, el 71% de la energía de la tasa es ese promedio y los clusters vuelven a ser de intensidad (alto, medio, bajo crecimiento, en curvas paralelas). Se resta el promedio de cada serie y no se reescala la amplitud: queda cómo varía el crecimiento alrededor de su propio ritmo, con explosiones grandes y chicas distinguibles.

In [ ]:
# ============================================================
# SERIES: tasa centrada, una matriz de 45 meses x 3 variables por artista
# ============================================================

# Tasa centrada por serie: se resta el promedio de cada artista y variable
panel[, crecimiento_centrado := crecimiento - mean(crecimiento), by = .(artist_id, variable)]

ancho <- dcast(panel[variable %in% VARIABLES_FORMA], artist_id + month ~ variable, value.var = "crecimiento_centrado")
filas_por_artista <- split(seq_len(nrow(ancho)), ancho$artist_id)
series_crecimiento <- lapply(filas_por_artista, function(filas) as.matrix(ancho[filas, ..VARIABLES_FORMA]))
series_crecimiento <- series_crecimiento[as.character(artistas$artist_id)]

# Sanity check: 45 x 3 por artista, sin NA y con media 0 por serie
stopifnot(
  length(series_crecimiento) == nrow(artistas),
  all(vapply(series_crecimiento, function(m) identical(dim(m), c(length(MESES_INTERIOR), length(VARIABLES_FORMA))), logical(1))),
  !anyNA(unlist(series_crecimiento)),
  max(abs(vapply(series_crecimiento, function(m) max(abs(colMeans(m))), numeric(1)))) < 1e-12
)

cat("Series:", length(series_crecimiento), "de", length(MESES_INTERIOR), "meses x", length(VARIABLES_FORMA), "variables\n")

## 3. Distancias: ¿ahora DTW ve algo que la euclídea no ve?

Con el nivel, euclídea y DTW casi coincidían (Spearman 0,998), porque las series crecían parejo. En la tasa, una explosión es un pico: si dos artistas tienen la misma explosión con unos meses de diferencia, la euclídea los separa y DTW (ventana de 6 meses) los alinea. Misma configuración de DTW que el notebook 04.

In [ ]:
# ============================================================
# MATRICES EUCLIDEA Y DTW SOBRE LA TASA DE CRECIMIENTO
# ============================================================

nombres_artistas <- names(series_crecimiento)

matriz_euclidea_forma <- as.matrix(dist(t(vapply(series_crecimiento, as.vector, numeric(length(series_crecimiento[[1]]))))))
matriz_dtw_forma <- as.matrix(proxy::dist(series_crecimiento, method = "dtw_basic", window.size = VENTANA_DTW_MESES,
                                          norm = NORMA_DTW, step.pattern = PATRON_PASOS_DTW))
dimnames(matriz_euclidea_forma) <- dimnames(matriz_dtw_forma) <- list(nombres_artistas, nombres_artistas)

# Sanity check: simétricas, diagonal 0, sin NA, y DTW <= euclídea
stopifnot(
  isSymmetric(unname(matriz_dtw_forma), tol = 1e-8),
  max(abs(diag(matriz_dtw_forma))) < 1e-8,
  !anyNA(matriz_dtw_forma),
  all(matriz_dtw_forma <= matriz_euclidea_forma + 1e-8)
)

In [ ]:
# ============================================================
# EUCLIDEA VS DTW: nivel (notebook 04) frente a forma del crecimiento
# ============================================================

triangulo <- function(m) m[upper.tri(m)]

vecinos_compartidos <- function(m1, m2, k = 10) {
  vecinos <- function(m) apply(m, 1, function(fila) order(fila)[2:(k + 1)])
  v1 <- vecinos(m1); v2 <- vecinos(m2)
  median(vapply(seq_len(ncol(v1)), function(i) length(intersect(v1[, i], v2[, i])), integer(1)))
}

resumen_euclidea_dtw <- function(m_euclidea, m_dtw) {
  cociente <- triangulo(m_dtw) / triangulo(m_euclidea)
  data.table(
    spearman = round(cor(triangulo(m_euclidea), triangulo(m_dtw), method = "spearman"), 3),
    vecinos_compartidos_de_10 = vecinos_compartidos(m_euclidea, m_dtw),
    cociente_mediano = round(median(cociente), 3),
    pct_pares_dtw_acerca_mas_de_18 = round(100 * mean(cociente < 0.82), 1)
  )
}

distancias_nivel <- readRDS(ARCHIVO_DISTANCIAS_NIVEL)$matrices
stopifnot(identical(rownames(distancias_nivel$dtw), nombres_artistas))

comparacion_representaciones <- rbind(
  data.table(representacion = "nivel (log1p centrado)", resumen_euclidea_dtw(distancias_nivel$euclidea, distancias_nivel$dtw)),
  data.table(representacion = "forma (tasa de crecimiento)", resumen_euclidea_dtw(matriz_euclidea_forma, matriz_dtw_forma))
)

comparacion_representaciones

In [ ]:
# ============================================================
# FIGURA: cociente DTW / euclídea en las dos representaciones
# Cerca de 1, DTW no cambia nada; más abajo, DTW encuentra a los
# dos artistas más parecidos al alinear eventos corridos.
# ============================================================

datos_cociente <- rbind(
  data.table(representacion = "Nivel", cociente = triangulo(distancias_nivel$dtw) / triangulo(distancias_nivel$euclidea)),
  data.table(representacion = "Tasa de crecimiento", cociente = triangulo(matriz_dtw_forma) / triangulo(matriz_euclidea_forma))
)

grafico_cociente <- ggplot(datos_cociente, aes(x = cociente, fill = representacion)) +
  geom_density(alpha = 0.6, colour = NA, adjust = 1.2) +
  geom_vline(xintercept = 1, colour = "#1D1E2C", linetype = "dashed") +
  scale_fill_manual(values = c("Nivel" = "#B8B8B8", "Tasa de crecimiento" = "#BC4B51"), name = NULL) +
  labs(
    title = "Donde DTW empieza a importar",
    subtitle = "Cociente DTW / euclídea para cada par de artistas (1 = DTW no cambia la distancia)",
    x = "DTW / euclídea", y = "Densidad"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top", panel.grid.minor = element_blank(),
        plot.background = element_rect(fill = "white", colour = NA))

print(grafico_cociente)

ggsave(file.path(FIGURES_DIR, "forma_01_cociente_dtw_euclidea.png"), grafico_cociente,
       width = 10, height = 5.5, dpi = 300, bg = "white")

In [ ]:
# ============================================================
# FIGURA PARA LA PRESENTACIÓN: DTW frente a la euclídea en la tasa
# Gemela de clustering_01 (notebook 04, nivel): misma escala de
# grises y diagonal blanca. En el nivel la nube se pega a la
# diagonal; en la tasa se despega hacia abajo.
# ============================================================

pares_dtw_vs_euclidea_forma <- data.table(
  euclidea = triangulo(matriz_euclidea_forma),
  dtw = triangulo(matriz_dtw_forma)
)

# Sanity check: DTW nunca supera a la euclídea (con ventana 0 coinciden)
stopifnot(pares_dtw_vs_euclidea_forma[, all(dtw <= euclidea + 1e-8)])

grafico_euclidea_vs_dtw_forma <- ggplot(pares_dtw_vs_euclidea_forma, aes(x = euclidea, y = dtw)) +
  geom_bin2d(bins = 120) +
  geom_abline(slope = 1, intercept = 0, colour = "white", linetype = "dashed", linewidth = 0.7) +
  scale_fill_gradientn(colours = c("#D9D9D9", "#9A9A9A", "#4A4A4A", "#111111"), trans = "log10",
                       name = "Pares de\nartistas",
                       labels = function(x) format(x, big.mark = ".", scientific = FALSE, trim = TRUE)) +
  coord_equal() +
  labs(
    title = "Ahora DTW sí acerca artistas",
    subtitle = paste0(
      "Cada punto es un par de artistas, en la tasa de crecimiento centrada.\n",
      "Bajo la diagonal, DTW los encuentra más parecidos (ventana de ", VENTANA_DTW_MESES, " meses)"
    ),
    x = "Distancia euclídea", y = "Distancia DTW"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    strip.text = element_text(face = "bold", size = 12, colour = "#111111", hjust = 0),
    axis.text = element_text(colour = "#6B6B6B", size = 10),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(colour = "#EDEDED"),
    panel.spacing = unit(18, "pt"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18),
    axis.title = element_text(face = "bold", colour = "#111111", size = 12),
    legend.title = element_text(size = 11, colour = "#4A4A4A"),
    legend.text = element_text(size = 10, colour = "#4A4A4A")
  )

ggsave(file.path(FIGURES_DIR, "forma_01b_euclidea_vs_dtw_tasa.png"), grafico_euclidea_vs_dtw_forma,
       width = 9, height = 8, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "forma_01b_euclidea_vs_dtw_tasa.png"))


In [ ]:
# ============================================================
# GUARDADO (archivo local, excluido de git)
# ============================================================

saveRDS(
  list(
    fecha_generacion = Sys.time(),
    archivo_panel = ARCHIVO_PANEL_PREPROCESADO,
    columna_serie = COLUMNA_SERIE,
    variables = VARIABLES_FORMA,
    meses = MESES_INTERIOR,
    ventana_dtw_meses = VENTANA_DTW_MESES,
    artistas = artistas,
    matrices = list(euclidea = matriz_euclidea_forma, dtw = matriz_dtw_forma)
  ),
  ARCHIVO_DISTANCIAS_FORMA
)

cat("Matrices guardadas en", ARCHIVO_DISTANCIAS_FORMA, "\n")

**Lectura de la etapa 1.** Con la tasa centrada, DTW sí importa: acerca más de un 18% al 10% de los pares (0,9% con el nivel) y cada artista comparte sólo 5 de sus 10 vecinos entre euclídea y DTW (8 con el nivel). Los corrimientos en el tiempo, invisibles en el nivel, aparecen en la forma del crecimiento.

---

## 4. ¿Se agrupan los artistas por cómo crecen? (etapa 2)

Spectral y PAM sobre la matriz DTW de la tasa, k de 2 a 8: los dos algoritmos que dieron perfiles legibles con el nivel, con la misma configuración que el notebook 05. El jerárquico queda afuera: con el nivel sólo separó casos extremos.

In [ ]:
# ============================================================
# CONFIGURACION DEL CLUSTERING (misma que el notebook 05)
# ============================================================

K_CANDIDATOS <- 2:8
VECINO_ESCALA_LOCAL <- 7L
INICIOS_KMEANS <- 25L
ARCHIVO_PARTICIONES_FORMA <- file.path("chartmetric_pilotos", "particiones_forma_crecimiento.rds")

SEMILLA_PRIMIGENIA <- 999199L

primos_seis_cifras <- function() {
  limite <- 999999L
  es_primo <- rep(TRUE, limite)
  es_primo[1] <- FALSE
  for (p in 2:floor(sqrt(limite))) {
    if (es_primo[p]) es_primo[seq(p * p, limite, by = p)] <- FALSE
  }
  candidatos <- which(es_primo)
  candidatos[candidatos >= 100000L]
}

PRIMOS_SEIS_CIFRAS <- primos_seis_cifras()

generar_semillas_primas <- function(n, semilla = SEMILLA_PRIMIGENIA) {
  set.seed(semilla)
  orden <- sample(length(PRIMOS_SEIS_CIFRAS))
  PRIMOS_SEIS_CIFRAS[orden[seq_len(n)]]
}

stopifnot(identical(generar_semillas_primas(3), c(742457L, 895957L, 543139L)))
SEMILLA_KMEANS_ESPECTRAL <- generar_semillas_primas(1)

In [ ]:
# ============================================================
# SPECTRAL Y PAM SOBRE DTW DE LA TASA DE CRECIMIENTO
# ============================================================

afinidad_escala_local <- function(m, vecino = VECINO_ESCALA_LOCAL) {
  escala <- apply(m, 1, function(fila) sort(fila)[vecino + 1])
  afinidad <- exp(-m^2 / outer(escala, escala))
  diag(afinidad) <- 0
  afinidad
}

descomposicion_espectral <- function(m) {
  afinidad <- afinidad_escala_local(m)
  grado <- rowSums(afinidad)
  eigen(afinidad / sqrt(outer(grado, grado)), symmetric = TRUE)
}

agrupar_espectral <- function(descomposicion, k, semilla = SEMILLA_KMEANS_ESPECTRAL) {
  incrustacion <- descomposicion$vectors[, seq_len(k), drop = FALSE]
  incrustacion <- incrustacion / sqrt(rowSums(incrustacion^2))
  set.seed(semilla)
  kmeans(incrustacion, centers = k, nstart = INICIOS_KMEANS, iter.max = 100)$cluster
}

descomposicion_forma <- descomposicion_espectral(matriz_dtw_forma)

particiones_forma <- rbind(
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = nombres_artistas, algoritmo = "espectral", k = k,
    cluster = agrupar_espectral(descomposicion_forma, k)
  ))),
  rbindlist(lapply(K_CANDIDATOS, function(k) data.table(
    artist_id = nombres_artistas, algoritmo = "pam", k = k,
    cluster = unname(pam(as.dist(matriz_dtw_forma), k = k, diss = TRUE)$clustering[nombres_artistas])
  )))
)

# Clusters renumerados por tamaño (1 = el más grande)
particiones_forma[
  ,
  cluster := match(as.character(cluster), names(sort(table(cluster), decreasing = TRUE))),
  by = .(algoritmo, k)
]

tamanios_forma <- particiones_forma[, .N, by = .(algoritmo, k, cluster)][
  order(algoritmo, k, cluster)][, .(artistas_por_cluster = paste(N, collapse = " / "), menor = min(N)), by = .(algoritmo, k)]

# Sanity check: todos los artistas una vez y k clusters por partición
stopifnot(particiones_forma[, .(n = .N, g = uniqueN(cluster)), by = .(algoritmo, k)][, all(n == length(nombres_artistas) & g == k)])

tamanios_forma

## 5. ¿Cuántas formas distintas hay? (etapa 3)

Mismos apoyos que con el nivel: eigengap y separación mínima entre los perfiles medianos (relativa a la distancia típica entre dos artistas). Los perfiles son la tasa centrada mediana de cada cluster: por encima de 0, el grupo crece más rápido que su propio ritmo; por debajo, más lento.

In [ ]:
# ============================================================
# EIGENGAP Y SEPARACION ENTRE PERFILES DE LA FORMA
# ============================================================

eigengap_forma <- data.table(
  k = K_CANDIDATOS,
  eigengap = round(descomposicion_forma$values[K_CANDIDATOS] - descomposicion_forma$values[K_CANDIDATOS + 1], 4)
)

matriz_tasas <- t(vapply(series_crecimiento, as.vector, numeric(length(series_crecimiento[[1]]))))
DISTANCIA_MEDIANA_ARTISTAS_FORMA <- median(matriz_euclidea_forma[upper.tri(matriz_euclidea_forma)])

perfil_mediano_clusters <- function(asignacion) {
  t(vapply(sort(unique(asignacion)), function(g)
    apply(matriz_tasas[asignacion == g, , drop = FALSE], 2, median), numeric(ncol(matriz_tasas))))
}

UMBRAL_CLUSTER_CHICO <- ceiling(0.02 * length(nombres_artistas))

separacion_forma <- particiones_forma[
  ,
  {
    asignacion <- cluster[match(nombres_artistas, artist_id)]
    .(separacion_minima = round(min(dist(perfil_mediano_clusters(asignacion))) / DISTANCIA_MEDIANA_ARTISTAS_FORMA, 2))
  },
  by = .(algoritmo, k)
]
separacion_forma <- merge(separacion_forma, tamanios_forma[, .(algoritmo, k, menor)], by = c("algoritmo", "k"))
separacion_forma[, cluster_chico := menor < UMBRAL_CLUSTER_CHICO]

cat("Eigengap (spectral):\n"); print(eigengap_forma)
cat("\nSeparación mínima entre perfiles:\n")
dcast(separacion_forma, algoritmo ~ k, value.var = "separacion_minima")

In [ ]:
# ============================================================
# FIGURA PARA LA PRESENTACIÓN: separación entre perfiles de la forma
# Gemela de clustering_04 (notebook 05): un panel por algoritmo,
# DTW en el mismo rojo. Escala propia por panel: PAM aísla grupos
# diminutos de saltos de YouTube y su separación llega a 5.
# ============================================================

ETIQUETAS_ALGORITMO_FORMA <- c(espectral = "Spectral", pam = "PAM")

datos_figura_separacion_forma <- copy(separacion_forma)
datos_figura_separacion_forma[, algoritmo_etiqueta := factor(ETIQUETAS_ALGORITMO_FORMA[algoritmo],
                                                             levels = ETIQUETAS_ALGORITMO_FORMA)]

grafico_separacion_forma <- ggplot(datos_figura_separacion_forma, aes(x = k, y = separacion_minima)) +
  geom_line(colour = "#C8102E", linewidth = 1.1) +
  geom_point(aes(shape = cluster_chico), colour = "#C8102E", size = 3, stroke = 1.2) +
  facet_wrap(~ algoritmo_etiqueta, nrow = 1, scales = "free_y") +
  scale_shape_manual(values = c(`FALSE` = 16, `TRUE` = 4), labels = c("No", "Sí"),
                     name = paste0("Cluster de menos de ", UMBRAL_CLUSTER_CHICO, " artistas")) +
  scale_x_continuous(breaks = K_CANDIDATOS) +
  expand_limits(y = 0) +
  labs(
    title = "¿Cuántas formas distintas hay?",
    subtitle = paste0(
      "Distancia entre los dos perfiles medianos más parecidos,\n",
      "relativa a la distancia típica entre dos artistas. DTW sobre la tasa"
    ),
    x = "k (cantidad de clusters)", y = "Separación mínima entre perfiles"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    strip.text = element_text(face = "bold", size = 12, colour = "#111111", hjust = 0),
    axis.title = element_text(colour = "#4A4A4A", size = 12),
    axis.text = element_text(colour = "#6B6B6B", size = 10),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(colour = "#EDEDED"),
    panel.spacing = unit(18, "pt"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18),
    legend.position = "bottom",
    legend.justification = "left",
    legend.title = element_text(size = 11, colour = "#4A4A4A"),
    legend.text = element_text(size = 11, colour = "#4A4A4A")
  )

ggsave(file.path(FIGURES_DIR, "forma_01c_separacion_formas.png"), grafico_separacion_forma,
       width = 11, height = 6, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "forma_01c_separacion_formas.png"))


In [ ]:
# ============================================================
# FIGURA: perfiles de la forma. Tasa de crecimiento mensual
# mediana por cluster, k de 2 a 5, spectral
# ============================================================

K_LECTURA_FORMA <- 2:5
PALETA_CLUSTERS <- c("#1982C4", "#F04A50", "#00A878", "#FF7A00", "#6A4C93", "#E4007C")
ETIQUETAS_FORMA <- VARIABLES_PANEL[VARIABLES_FORMA]

perfiles_forma_largo <- function(algoritmo_elegido, ks) {
  rbindlist(lapply(ks, function(k_elegido) {
    asignacion <- particiones_forma[algoritmo == algoritmo_elegido & k == k_elegido][match(nombres_artistas, artist_id), cluster]
    perfiles <- perfil_mediano_clusters(asignacion)
    grupos <- sort(unique(asignacion))
    data.table(
      k = k_elegido,
      cluster = rep(grupos, times = ncol(perfiles)),
      artistas = rep(as.vector(table(asignacion)[as.character(grupos)]), times = ncol(perfiles)),
      variable = rep(VARIABLES_FORMA, each = length(MESES_INTERIOR) * length(grupos)),
      month = rep(rep(MESES_INTERIOR, each = length(grupos)), times = length(VARIABLES_FORMA)),
      crecimiento_pct = 100 * as.vector(perfiles)
    )
  }))
}

grafico_perfiles_forma <- function(algoritmo_elegido) {
  datos <- perfiles_forma_largo(algoritmo_elegido, K_LECTURA_FORMA)
  datos[, `:=`(
    variable_etiqueta = factor(ETIQUETAS_FORMA[variable], levels = ETIQUETAS_FORMA),
    k_etiqueta = paste0("k = ", k),
    cluster_etiqueta = factor(cluster)
  )]
  ggplot(datos, aes(x = month, y = crecimiento_pct, colour = cluster_etiqueta)) +
    geom_hline(yintercept = 0, colour = "grey75") +
    geom_line(linewidth = 1) +
    facet_grid(k_etiqueta ~ variable_etiqueta, scales = "free_y") +
    scale_colour_manual(values = PALETA_CLUSTERS, name = "Cluster") +
    scale_x_date(date_breaks = "1 year", date_labels = "%Y") +
    labs(
      title = paste0("Formas del crecimiento: ", ifelse(algoritmo_elegido == "espectral", "Spectral", "PAM"), " con DTW"),
      subtitle = "Tasa de crecimiento mensual centrada, mediana por cluster (puntos porcentuales respecto del ritmo propio). Cluster 1 = el más grande",
      x = NULL, y = "Crecimiento respecto del ritmo propio (p. p.)"
    ) +
    theme_minimal(base_size = 11) +
    theme(panel.grid.minor = element_blank(), strip.text = element_text(face = "bold"),
          legend.position = "bottom", plot.background = element_rect(fill = "white", colour = NA))
}

for (algoritmo_elegido in c("espectral", "pam")) {
  grafico <- grafico_perfiles_forma(algoritmo_elegido)
  print(grafico)
  ggsave(file.path(FIGURES_DIR, paste0("forma_02_perfiles_", algoritmo_elegido, ".png")), grafico,
         width = 13, height = 10, dpi = 300, bg = "white")
}

In [ ]:
# ============================================================
# FIGURA PARA LA PRESENTACIÓN: las tres formas del crecimiento
# Spectral con DTW, k = 3. Colores de línea de la forma, los mismos
# del notebook 07: parejo naranja, enfriamiento azul, salto en
# YouTube verde.
# Nombres: el grupo más grande es parejo; entre los otros dos, el
# del pico mayor de suscriptores de YouTube es el salto.
# ============================================================

perfiles_forma_k3 <- perfiles_forma_largo("espectral", 3)

tamanio_por_cluster <- unique(perfiles_forma_k3[, .(cluster, artistas)])
pico_youtube <- perfiles_forma_k3[variable == "youtube_channel_subscribers", .(pico = max(crecimiento_pct)), by = cluster]
nombres_forma <- merge(tamanio_por_cluster, pico_youtube, by = "cluster")
cluster_parejo <- nombres_forma[which.max(artistas), cluster]
cluster_salto <- nombres_forma[cluster != cluster_parejo][which.max(pico), cluster]
nombres_forma[, perfil := fifelse(cluster == cluster_parejo, "Parejo",
                          fifelse(cluster == cluster_salto, "Salto en YouTube", "Enfriamiento"))]
nombres_forma[, perfil := factor(perfil, levels = c("Parejo", "Enfriamiento", "Salto en YouTube"))]
setorder(nombres_forma, perfil)
nombres_forma[, perfil_etiqueta := paste0(perfil, " (", artistas, ")")]

# Sanity check: tres nombres distintos
stopifnot(uniqueN(nombres_forma$perfil) == 3)
nombres_forma

COLORES_FORMA <- setNames(c("#E0A100", "#2F55C9", "#1E9E5A"), nombres_forma$perfil_etiqueta)

perfiles_forma_k3 <- merge(perfiles_forma_k3, nombres_forma[, .(cluster, perfil_etiqueta)], by = "cluster")
perfiles_forma_k3[, `:=`(
  variable_etiqueta = factor(ETIQUETAS_FORMA[variable], levels = ETIQUETAS_FORMA),
  perfil_etiqueta = factor(perfil_etiqueta, levels = nombres_forma$perfil_etiqueta)
)]

grafico_formas_k3 <- ggplot(perfiles_forma_k3, aes(x = month, y = crecimiento_pct, colour = perfil_etiqueta)) +
  geom_hline(yintercept = 0, colour = "#BDBDBD", linewidth = 0.4) +
  geom_line(linewidth = 1.3) +
  facet_wrap(~ variable_etiqueta, nrow = 1, scales = "free_y") +
  scale_colour_manual(values = COLORES_FORMA, name = "Perfil (artistas)") +
  scale_x_date(date_breaks = "1 year", date_labels = "%Y") +
  labs(
    title = "Tres formas del crecimiento",
    subtitle = paste0(
      "Tasa de crecimiento mensual centrada, mediana por perfil (spectral con DTW, k = 3).\n",
      "0 = el grupo creciendo a su propio ritmo. Cada panel tiene su escala"
    ),
    x = NULL, y = "Crecimiento mensual respecto\ndel ritmo propio (p. p.)"
  ) +
  theme_minimal(base_size = 13, base_family = "LWR Mono") +
  theme(
    plot.title = element_text(family = "LWR Titulo", size = 26, colour = "#111111", margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#4A4A4A", lineheight = 1.15, margin = margin(b = 14)),
    strip.text = element_text(face = "bold", size = 12, colour = "#111111", hjust = 0),
    axis.title = element_text(colour = "#4A4A4A", size = 12),
    axis.text = element_text(colour = "#6B6B6B", size = 10),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(colour = "#EDEDED"),
    panel.spacing = unit(18, "pt"),
    plot.title.position = "plot",
    plot.background = element_rect(fill = "white", colour = NA),
    plot.margin = margin(18, 18, 12, 18),
    legend.position = "bottom",
    legend.justification = "left",
    legend.title = element_text(size = 12, colour = "#4A4A4A"),
    legend.text = element_text(face = "bold", size = 12, colour = "#111111"),
    legend.key.width = unit(34, "pt")
  )

ggsave(file.path(FIGURES_DIR, "forma_02b_perfiles_forma_k3.png"), grafico_formas_k3,
       width = 12, height = 6, dpi = 300, bg = "white", device = ragg::agg_png)

# El dispositivo del notebook no conoce las tipografías registradas:
# se muestra el PNG guardado
IRdisplay::display_png(file = file.path(FIGURES_DIR, "forma_02b_perfiles_forma_k3.png"))


In [ ]:
# ============================================================
# GUARDADO DE LAS PARTICIONES DE LA FORMA (local, fuera de git)
# ============================================================

saveRDS(
  list(
    fecha_generacion = Sys.time(),
    configuracion = list(k_candidatos = K_CANDIDATOS, vecino_escala_local = VECINO_ESCALA_LOCAL,
                         inicios_kmeans = INICIOS_KMEANS, semilla_kmeans_espectral = SEMILLA_KMEANS_ESPECTRAL),
    artistas = artistas,
    particiones = particiones_forma,
    tamanios = tamanios_forma,
    eigengap = eigengap_forma,
    separacion = separacion_forma
  ),
  ARCHIVO_PARTICIONES_FORMA
)

cat("Particiones guardadas en", ARCHIVO_PARTICIONES_FORMA, "\n")

## Lectura de las etapas 2 y 3 (a validar con la estabilidad, notebook 07)

- **k = 3 con spectral.** La separación entre perfiles se mantiene alta en k = 2 y 3 (0,87) y cae en k = 4 (0,13): el cuarto grupo repite una forma.
  1. **Parejos** (1053): crecen a su propio ritmo durante todo el período.
  2. **Enfriamiento** (173): crecían por encima de su ritmo en 2022 y 2023 y se frenan con fuerza desde 2024, en las tres plataformas a la vez. En una audiencia acumulada, es la versión del "explosión y caída".
  3. **Salto en YouTube** (26): un pico aislado de suscriptores en 2024, del tipo de un cambio de canal en la fuente. A revisar como anomalía.
- **PAM** aísla primero esos saltos de YouTube y deja una masa de más del 90%: para la forma, spectral es el algoritmo principal.
- La mayoría de los artistas no tiene una forma distintiva: crece a su ritmo y desacelera suave.